# F4-0：读取成本的拆解

f4 设计第 3 节（`docs/experiments/f4/2026-10-25-f4-design.md`）。A3、A0、P 三组读者在 k10、k20、k30 上的成本，从现有轨迹拆开，不新跑。

- 数据：`~/pw-lab/runs/f3/analysis/f4-0/`（仓库外），由 `pweval.measure.read_cost` 产出；口径与核对见该模块的说明。重新生成（仓库根目录）：

  ```
  uv run --frozen python -m pweval.measure.read_cost f3:A3 f3:A0 ref1:P
  ```

- 本 notebook 只读这些表，做切片与作图；图存到同一目录的 `figs/`。
- **题是嵌套的**：k10 的 53 题都在 k20 的 69 题中，69 题都在 k30 的 200 题中。所以每张图画两种：**同一批 53 题**（题不变、库变大，看规模的效应），与**各检查点的全部题**（题的构成也在变：k30 新加的多是跨论文题）。
- 每个点是每题的平均值，竖线是按题自助抽样的 95% 区间（2000 次，种子固定）。三组的题号相同，每题都有三组的成本。
- 不含辅助题。成本照计上下文溢出中断的会话（与读者记录的口径相同）。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import pandas as pd
import yaml

DATA = Path.home() / "pw-lab/runs/f3/analysis/f4-0"
FIGS = DATA / "figs"
FIGS.mkdir(exist_ok=True)
meta = yaml.safe_load((DATA / "meta.yml").read_text())
sessions = pd.read_csv(DATA / "sessions.csv")
tools = pd.read_csv(DATA / "tools.csv")
print(meta["created"], meta["code"]["commit"][:7], "read_cost dirty:", meta["code"]["read_cost_dirty"])
print(meta["rows"], "chars/token (fallback only):", meta["chars_per_token"])

## 图的约定

三组用参考色板的前三个分类色，固定对应（A3 蓝、A0 橙、P 青），所有图一致；这三色在全部两两组合下通过色觉校验。青色对浅色背景不足 3:1，所以每条线的末端有文字标注，每张图后附数值表。每张小图只有一个纵轴；量纲不同的指标分开画。

In [ ]:
ARMS = ["A3", "A0", "P"]
COLOR = {"A3": "#2a78d6", "A0": "#eb6834", "P": "#1baf7a"}
LABEL = {"A3": "A3 (middleware, MCP)", "A0": "A0 (files + Git)", "P": "P (full text)"}
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
KS = [10, 20, 30]
DODGE = {"A3": -0.5, "A0": 0.0, "P": 0.5}
CROSS = {"T3", "T7", "T8"}
SINGLE = {"T9", "T10", "T11", "T14"}
FIXED = set(sessions[(sessions.k == 10) & (sessions.arm == "A3")].probe)
assert all(FIXED <= set(sessions[(sessions.k == k) & (sessions.arm == a)].probe) for k in KS for a in ARMS)

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9.5, "axes.titleweight": "bold",
    "axes.edgecolor": GRID, "axes.linewidth": 1, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.axisbelow": True,
})
RNG_SEED = 20261025


def boot_ci(values, n=2000, seed=RNG_SEED):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), (n, len(v)))].mean(axis=1)
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def subset(df, which):
    if which == "fixed":
        return df[df.probe.isin(FIXED)]
    return df


def per_question(frame, value, which, arms=ARMS, stat="mean"):
    # 每组、每个检查点：每题平均值与 95% 区间（stat='median' 时只给中位数）
    out = []
    for arm in arms:
        for k in KS:
            v = subset(frame[(frame.arm == arm) & (frame.k == k)], which)[value]
            if stat == "median":
                m, lo, hi = v.median(), np.nan, np.nan
            else:
                m, lo, hi = boot_ci(v)
            out.append({"arm": arm, "k": k, "n": int(v.notna().sum()), "value": m, "lo": lo, "hi": hi})
    return pd.DataFrame(out)


def end_labels(ax, points, fmt):
    # 线末端的文字标注（墨色，不用系列色）；标签相互挤在一起时上下错开，用细引线连回线端
    if not points:
        return
    lo, hi = ax.get_ylim()
    gap = (hi - lo) * 0.075
    placed = []
    for arm, y in sorted(points, key=lambda p: p[1]):
        ty = max(y, placed[-1][1] + gap) if placed else y
        placed.append((arm, ty, y))
    shift = max(0.0, placed[-1][1] - hi + gap / 2)
    for arm, ty, y in placed:
        ty -= shift
        x0 = KS[-1] + DODGE[arm]
        ax.annotate(f"{arm} {fmt(y)}", xy=(x0, y), xytext=(KS[-1] + 2.2, ty), color=INK, fontsize=8, va="center",
                    arrowprops={"arrowstyle": "-", "color": INK2, "lw": 0.6, "shrinkA": 0, "shrinkB": 3})


def line_panel(ax, table, title, ylabel, fmt=lambda v: f"{v:,.0f}", arms=ARMS):
    points = []
    for arm in arms:
        t = table[table.arm == arm].sort_values("k")
        if t.value.isna().all():
            continue
        x = t.k + DODGE[arm]
        ax.plot(x, t.value, color=COLOR[arm], lw=2, solid_capstyle="round", solid_joinstyle="round", zorder=3)
        ax.vlines(x, t.lo, t.hi, color=COLOR[arm], lw=1, alpha=0.55, zorder=2)
        ax.plot(x, t.value, "o", ms=6.5, color=COLOR[arm], mec=SURFACE, mew=1.6, zorder=4)
        points.append((arm, float(t.value.iloc[-1])))
    ax.set_title(title, loc="left", color=INK)
    ax.set_ylabel(ylabel)
    ax.set_xticks(KS, [f"k{k}" for k in KS])
    ax.set_xlim(7, 38)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    end_labels(ax, points, fmt)


def figure(panels, which, suptitle, name, ncols=None):
    # panels: [(table_fn, title, ylabel, fmt, arms)]
    ncols = ncols or len(panels)
    nrows = int(np.ceil(len(panels) / ncols))
    height = 2.7 * nrows + 0.7  # 0.7 英寸给标题与图例
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, height), squeeze=False)
    tables = {}
    for ax, (fn, title, ylabel, fmt, arms) in zip(axes.flat, panels):
        t = fn(which)
        tables[title] = t
        line_panel(ax, t, title, ylabel, fmt, arms)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    shown = [a for a in ARMS if any(a in arms for *_, arms in panels)]
    handles = [Line2D([], [], color=COLOR[a], lw=2, marker="o", ms=6.5, mec=SURFACE, mew=1.6) for a in shown]
    fig.suptitle(suptitle, x=0.01, y=1 - 0.08 / height, ha="left", va="top", fontsize=10.5, fontweight="bold", color=INK)
    fig.legend(handles, [LABEL[a] for a in shown], loc="upper left", bbox_to_anchor=(0.005, 1 - 0.36 / height), ncol=3,
               frameon=False, fontsize=8.5, handlelength=2.2, labelcolor=INK)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.7 / height))
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}-{which}.{ext}", dpi=160, bbox_inches="tight")
    plt.show()
    return tables


def show_tables(tables, scale=1.0, digits=0):
    rows = []
    for title, t in tables.items():
        w = t.assign(v=lambda d: (d.value / scale).round(digits)).pivot(index="arm", columns="k", values="v")
        w = w.reindex([a for a in ARMS if a in w.index])
        w.columns = [f"k{k}" for k in w.columns]
        w.insert(0, "panel", title)
        rows.append(w.reset_index())
    return pd.concat(rows, ignore_index=True)

WHICH = {"fixed": "same 53 questions at every checkpoint", "all": "all questions of each checkpoint (53 / 69 / 200)"}

## 1 每题的总成本

用时、未缓存输入、缓存读与模型调用次数，每题平均。f3 读者记录的合计口径不同（含辅助题、按合计报），这里按每题平均，便于跨检查点比较。

In [ ]:
s = sessions.assign(seconds_min=sessions.seconds / 60, input_k=sessions.input / 1e3, cache_k=sessions.cache_read / 1e3)


def col(value):
    return lambda which: per_question(s, value, which)


TOTAL = [
    (col("seconds_min"), "Time per question", "minutes", lambda v: f"{v:.1f}", ARMS),
    (col("input_k"), "Uncached input per question", "k tokens", lambda v: f"{v:.0f}k", ARMS),
    (col("cache_k"), "Cache reads per question", "k tokens", lambda v: f"{v:,.0f}k", ARMS),
    (col("calls"), "Model calls per question", "calls", lambda v: f"{v:.1f}", ARMS),
]
total_tables = {w: figure(TOTAL, w, f"Reader cost per question, {WHICH[w]}", "f4-0-total") for w in WHICH}

In [ ]:
show_tables(total_tables["fixed"], digits=1)

In [ ]:
show_tables(total_tables["all"], digits=1)

## 2 提示 token 的来源

每题提示 token（input 加 cacheRead）拆成驻留成本：固定前缀（第一次调用的提示乘以调用次数）、助手输出、查找类工具的返回、`read_evidence` 的返回、残差（只在压缩过的会话中出现）。查找类：A3 为 search、get、traverse、log、show、checkout、as_of 与它少量的 bash 和 read；A0、P 为 bash 与 pi 的 read。P 没有 `read_evidence`，它的原文读取都在查找类里。

In [ ]:
LOOKUP = tools[tools.tool != "read_evidence"].groupby(["arm", "k", "probe"]).occupancy.sum().rename("occ_lookup")
EVID = tools[tools.tool == "read_evidence"].groupby(["arm", "k", "probe"]).occupancy.sum().rename("occ_evidence")
src = s.set_index(["arm", "k", "probe"]).join(LOOKUP).join(EVID).fillna({"occ_lookup": 0, "occ_evidence": 0}).reset_index()
assert ((src.occ_lookup + src.occ_evidence) - src.occ_tools).abs().max() < 1
for c in ["prompt_total", "occ_prefix", "occ_output", "occ_lookup", "occ_evidence", "occ_residual"]:
    src[c + "_k"] = src[c] / 1e3


def scol(value, arms=ARMS):
    return lambda which: per_question(src, value, which, arms)


K = lambda v: f"{v:,.0f}k"
SOURCES = [
    (scol("prompt_total_k"), "All prompt tokens", "k tokens", K, ARMS),
    (scol("occ_prefix_k"), "Fixed prefix", "k tokens", K, ARMS),
    (scol("occ_output_k"), "Assistant output", "k tokens", K, ARMS),
    (scol("occ_lookup_k"), "Lookup tool returns", "k tokens", K, ARMS),
    (scol("occ_evidence_k"), "read_evidence returns", "k tokens", K, ["A3", "A0"]),
    (scol("occ_residual_k"), "Residual (compacted sessions)", "k tokens", K, ARMS),
]
source_tables = {w: figure(SOURCES, w, f"Prompt tokens per question by source, {WHICH[w]}", "f4-0-sources", ncols=3)
                 for w in WHICH}

In [ ]:
show_tables(source_tables["fixed"])

In [ ]:
show_tables(source_tables["all"])

## 3 查找的形状：步数、单步大小、定位与固定开销

- 每题的查找调用次数与单次返回的字符数（中位数，所有查找调用合在一起）：看"步数多、每步小"还是"步数少、每步大"。
- 定位成本：第一次成功的 `read_evidence` 之前的模型调用次数（P 不适用）。
- 第一次调用的提示 token：系统提示、工具定义与题面；A0 含 pi 载入的 `AGENTS.md`，A3 含 MCP 工具定义。

后两项以中位数报，没有区间。

In [ ]:
look = tools[tools.tool != "read_evidence"]
look_calls = look.groupby(["arm", "k", "probe"]).size().rename("lookup_calls")
shape = s.set_index(["arm", "k", "probe"]).join(look_calls).fillna({"lookup_calls": 0}).reset_index()


def median_chars(which):
    out = []
    for arm in ARMS:
        for k in KS:
            v = subset(look[(look.arm == arm) & (look.k == k)], which).chars
            out.append({"arm": arm, "k": k, "n": len(v), "value": v.median(), "lo": np.nan, "hi": np.nan})
    return pd.DataFrame(out)


SHAPE = [
    (lambda w: per_question(shape, "lookup_calls", w), "Lookup calls per question", "calls", lambda v: f"{v:.1f}", ARMS),
    (median_chars, "Chars per lookup return (median)", "characters", lambda v: f"{v:,.0f}", ARMS),
    (lambda w: per_question(s, "calls_before_evidence", w, ["A3", "A0"], stat="median"),
     "Calls before first evidence (median)", "calls", lambda v: f"{v:.0f}", ["A3", "A0"]),
    (lambda w: per_question(s, "prompt_first", w, stat="median"), "First prompt (median)", "tokens",
     lambda v: f"{v:,.0f}", ARMS),
]
shape_tables = {w: figure(SHAPE, w, f"Shape of lookup, {WHICH[w]}", "f4-0-shape") for w in WHICH}

In [ ]:
show_tables(shape_tables["fixed"], digits=1)

In [ ]:
show_tables(shape_tables["all"], digits=1)

## 4 按题型：跨论文题与单篇查找题

每题提示 token，按各检查点的全部题。跨论文题为 T3、T7、T8，单篇查找题为 T9、T10、T11、T14，其余题型合为一类。每个点的题数见表中的 `n`；k10、k20 的跨论文题较少，区间较宽。

In [ ]:
fam = src.assign(family=np.where(src.template.isin(CROSS), "cross", np.where(src.template.isin(SINGLE), "single", "other")))


def fam_col(family):
    return lambda which: per_question(fam[fam.family == family], "prompt_total_k", which)


FAMILY = [
    (fam_col("cross"), "Cross-paper (T3, T7, T8)", "k tokens", K, ARMS),
    (fam_col("single"), "Single-paper lookup (T9-T11, T14)", "k tokens", K, ARMS),
    (fam_col("other"), "Other templates", "k tokens", K, ARMS),
]
family_tables = figure(FAMILY, "all", "Prompt tokens per question by template family, all questions", "f4-0-family")
pd.concat({t: v.set_index(["arm", "k"])[["n", "value", "lo", "hi"]].round(0) for t, v in family_tables.items()}, axis=1)

## 5 压缩过的会话

残差只出现在压缩过的会话中（压缩摘要与保留下来的末段不归到具体来源）。各组、各检查点压缩过的会话数：

In [ ]:
sessions.assign(compacted=sessions.compactions > 0).groupby(["arm", "k"]).agg(
    questions=("probe", "size"), compacted=("compacted", "sum"), stop_error=("stop", lambda x: (x == "error").sum())
).unstack("k")